# One agent, the whole pipeline

Every notebook so far called one function at a time. Here, one agent gets
all of it as tools - extraction, comparison, questions, bias checks, and
interview scheduling - and decides which to call and in what order, the same
combined-agent idea as `cms-app`'s notebook 04.

The one rule that is non-negotiable: the agent can *propose* interview
times, but it can never *book* one without a human confirming which slot
first. That is built as two separate tools, not one tool that books
automatically - "review important actions before they are executed", from
the spec, made structural rather than just requested in a prompt.

## Step 1 - where candidates and the JD come from

If `GOOGLE_DRIVE_JD_FOLDER_ID` / `GOOGLE_DRIVE_RESUMES_FOLDER_ID` are set,
this pulls the live JD and resumes from Drive - edit the JD there or drop in
a new resume, and the next tool call picks it up. Otherwise it falls back to
`sample_data/`, so this notebook also runs with zero Google setup.

In [ ]:
import os, getpass
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
if not os.environ.get("LITELLM_API_KEY"):
    os.environ["LITELLM_API_KEY"] = getpass.getpass("LiteLLM API key: ")

from recruiting import (
    extract_candidate,
    extract_job_description,
    evaluate_candidate,
    rank_candidates,
    generate_questions,
    apply_bias_check,
    CalendarClient,
)

USE_DRIVE = bool(os.environ.get("GOOGLE_DRIVE_JD_FOLDER_ID"))
SAMPLE_DIR = Path("sample_data")

if USE_DRIVE:
    from recruiting import DriveClient

    drive = DriveClient()
    jd_folder = os.environ["GOOGLE_DRIVE_JD_FOLDER_ID"]
    resumes_folder = os.environ["GOOGLE_DRIVE_RESUMES_FOLDER_ID"]

    def _load_jd_text():
        f = drive.latest(jd_folder)
        return drive.fetch_text(f), f.name

    def _load_resume_texts():
        return [(drive.fetch_text(f), f.name) for f in drive.list_files(resumes_folder)]
else:

    def _load_jd_text():
        path = SAMPLE_DIR / "job_description.txt"
        return path.read_text(encoding="utf-8"), path.name

    def _load_resume_texts():
        return [
            (p.read_text(encoding="utf-8"), p.name) for p in sorted(SAMPLE_DIR.glob("resume_*.txt"))
        ]

print("Source:", "Google Drive" if USE_DRIVE else "sample_data/")

## Step 2 - the tools

A small in-memory cache so re-running a tool during a conversation does not
re-call the LLM for a resume it already read. `list_candidates` always
re-lists the source, so a new resume or an edited JD is picked up on the
next call - that is how this design handles the "JD changed" / "new resume
submitted" live-demo cases, with no special-casing.

In [ ]:
from langchain.tools import tool

_evaluations = {}
_last_slots = []


def _refresh():
    jd_text, jd_name = _load_jd_text()
    jd = extract_job_description(jd_text, source_file=jd_name)
    for text, name in _load_resume_texts():
        if name in _evaluations and _evaluations[name].candidate.raw_text == text:
            continue  # unchanged since last time, keep the cached evaluation
        candidate = extract_candidate(text, source_file=name)
        _evaluations[name] = evaluate_candidate(candidate, jd)
    return jd


@tool
def list_candidates() -> list[dict]:
    """List every candidate for the current JD, ranked best first, with score
    and whether they're tied with the next candidate."""
    _refresh()
    ranked = rank_candidates(list(_evaluations.values()))
    return [
        {"name": e.candidate.name, "score": e.score, "tied_with_next": e.tied_with_next}
        for e in ranked
    ]


@tool
def get_candidate_evidence(name: str) -> dict:
    """Full per-requirement evidence for one candidate: matched/not, quotes,
    transferable-skill notes, missing info, inconsistencies. Call this before
    explaining why someone is or isn't a fit."""
    e = next((v for v in _evaluations.values() if name.lower() in v.candidate.name.lower()), None)
    if not e:
        return {"error": f"no candidate matches {name!r}"}
    return {
        "candidate": e.candidate.name,
        "score": e.score,
        "matches": [m.__dict__ for m in e.matches],
        "missing_info": e.missing_info,
        "inconsistencies": e.inconsistencies,
    }


@tool
def generate_interview_questions_for(name: str) -> list[dict]:
    """Personalized interview questions for one candidate, grounded in their
    specific gaps and transferable-skill matches."""
    e = next((v for v in _evaluations.values() if name.lower() in v.candidate.name.lower()), None)
    if not e:
        return [{"error": f"no candidate matches {name!r}"}]
    return [q.__dict__ for q in generate_questions(e)]


@tool
def check_bias_for(name: str) -> list[str]:
    """Re-check one candidate's evaluation reasoning for bias. Run this before
    finalizing a recommendation."""
    e = next((v for v in _evaluations.values() if name.lower() in v.candidate.name.lower()), None)
    if not e:
        return [f"no candidate matches {name!r}"]
    apply_bias_check(e)
    return e.bias_flags


@tool
def propose_interview_slots(interviewer_emails: list[str], duration_min: int = 45) -> list[str]:
    """Propose open interview slots across every interviewer's calendar. This
    only reads calendars - it never books anything. Always call this and show
    the options to the recruiter before ever calling book_interview."""
    global _last_slots
    cal = CalendarClient(calendar_id=os.environ.get("GOOGLE_CALENDAR_ID", "primary"))
    _last_slots = cal.find_free_slots(interviewer_emails, duration_min=duration_min)
    if not _last_slots:
        return ["No open slots in the next few days - try widening the search."]
    return [f"{i}: {s}" for i, s in enumerate(_last_slots)]


@tool
def book_interview(candidate_name: str, slot_index: int, interviewer_emails: list[str]) -> str:
    """Book the interview at the given slot index from the most recent
    propose_interview_slots call. ONLY call this after the recruiter has
    explicitly confirmed a slot in the conversation - never on your own
    initiative."""
    if not (0 <= slot_index < len(_last_slots)):
        return "That slot index isn't from the most recent proposal - call propose_interview_slots again."
    cal = CalendarClient(calendar_id=os.environ.get("GOOGLE_CALENDAR_ID", "primary"))
    link = cal.create_event(
        summary=f"Interview: {candidate_name}",
        slot=_last_slots[slot_index],
        attendee_emails=interviewer_emails,
    )
    return f"Booked. {link}"

## Step 3 - the agent

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

from recruiting import get_model

agent = create_agent(
    model=get_model(),
    tools=[
        list_candidates,
        get_candidate_evidence,
        generate_interview_questions_for,
        check_bias_for,
        propose_interview_slots,
        book_interview,
    ],
    system_prompt=(
        "You help a recruiter screen candidates for one open role. Always call "
        "list_candidates first to see who's in play. Before explaining why "
        "someone is or isn't a fit, call get_candidate_evidence - never assert "
        "a match without it. If two candidates are close in score, say so "
        "explicitly instead of picking a favorite. Before finalizing a "
        "recommendation, call check_bias_for on it. "
        "For scheduling: always call propose_interview_slots and show the "
        "options to the recruiter first. Only call book_interview after the "
        "recruiter has explicitly picked a slot in this conversation - never "
        "book on your own initiative."
    ),
)


def ask(question):
    result = agent.invoke({"messages": [HumanMessage(question)]})
    return result["messages"][-1].content


print(ask(
    "Who's the strongest candidate for this role, and why? "
    "Also check the evaluation for bias before you answer."
))

## Step 4 - the live-demo curveballs

Each bullet from the spec's "Possible Live-Demo Changes", and how this
design already handles it:

- **A JD requirement changes** - edit the JD (in Drive, or
  `sample_data/job_description.txt`) and ask again. `list_candidates` always
  calls `_refresh()`, which re-reads the source, so the ranking updates on
  the next question with no special-casing.
- **A candidate submits a new resume** - drop the file into the resumes
  folder (or `sample_data/`) and ask "any new candidates?". Same mechanism.
- **An interviewer becomes unavailable** - `propose_interview_slots` reads
  live free/busy data, so their busy time is already excluded. If everyone
  is fully booked it says so, and the system prompt has the agent suggest
  widening the search rather than inventing a slot.
- **Two candidates have similar qualifications** - `rank_candidates` sets
  `tied_with_next`, and the system prompt tells the agent to name the tie
  instead of confidently picking one.
- **The system detects potentially biased evaluation criteria** -
  `check_bias_for` runs before any recommendation is finalized, per the
  system prompt.

In [ ]:
print(ask(
    "Are any two candidates close enough in score that I should interview "
    "both instead of picking one?"
))

## Step 5 - scheduling, with a human in the loop

This needs `GOOGLE_CALENDAR_ID` and real interviewer emails with calendars -
skip this cell if you have not set that up. Notice the agent proposes, then
stops and waits; it only books after you say which slot in a follow-up
message.

In [ ]:
print(ask(
    "Propose interview times for Amina Hassan with "
    "interviewer1@example.com and interviewer2@example.com, 45 minutes."
))

## Step 6 - confirming a slot

Only after you have looked at the options above and picked one:

In [ ]:
print(ask("Book the second option for Amina Hassan."))

## What just happened, and the point

One agent, six tools, and one hard rule enforced by design rather than by
asking nicely: proposing a calendar slot and booking it are two different
tools, and the system prompt (backed by `book_interview`'s own docstring)
never lets the second run without a human picking from the first.

Your turn:

- Point `GOOGLE_DRIVE_JD_FOLDER_ID` at a real Drive folder, edit the JD
  there, and ask the agent to re-rank candidates.
- The click-through version of this same pipeline is `recruiter_app.py`
  (`uv run streamlit run recruiter_app.py`) - same package underneath, a UI
  instead of a chat.